#### LLaVA Next model on Transformers API with MH-Post_Hoc Framework
1. Loading LLaVA Next model on GPU0
2. Loading MH-Detector on GPU1
3. Building the refinement loop and Ranking mechanism 
4. Sorting intermediate results, including generated content and similarity score, and final result
5. Converting the final result into Amber Json file format

In [ ]:
import requests
from PIL import Image
import os
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
# Set Hugging Face cache directory
os.environ["HF_HOME"] = "/home/ubuntu/project/cache/huggingface"

model = AutoModelForImageTextToText.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf", torch_dtype=torch.float16, device_map={"": "cuda:0"})
processor = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")

# Load CLIP model on GPU1
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda:1")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("LLaVA Model loaded on GPU0")
print("CLIP Model loaded on GPU1")


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.2k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


NameError: name 'CLIPModel' is not defined

In [3]:
from transformers import CLIPProcessor, CLIPModel
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda:1")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("LLaVA Model loaded on GPU0")
print("CLIP Model loaded on GPU1")


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

LLaVA Model loaded on GPU0
CLIP Model loaded on GPU1


In [6]:
import os
import json
from PIL import Image
import requests
import random
llava_processor = processor
# Path to the dataset and JSON file
dataset_dir = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/AMBER/image"
query_generative_json_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/AMBER/data/query/query_generative.json"

# Load the generative queries JSON file
with open(query_generative_json_path, "r") as f:
    query_generative_json = json.load(f)

# Check the number of queries in the query_generative.json
print("Number of queries in the query_generative.json: ", len(query_generative_json))

# Randomly select a few queries for testing purposes (e.g., 10)
random.seed(0)
selected_queries = random.sample(query_generative_json, 10)
print("Selected queries: ", selected_queries)

# Load images and prompts
images = []
prompts = []

for query in selected_queries:
    # Get the image path
    image_filename = query['image']
    image_path = os.path.join(dataset_dir, image_filename)

    # Load the image
    if os.path.exists(image_path):
        image = Image.open(image_path)
        images.append(image)
    else:
        print(f"Image not found: {image_path}")
        continue

    # Prepare the prompt
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": query['query']},
            ],
        },
    ]
    prompts.append(prompt)

print(f"Loaded {len(images)} images and {len(prompts)} prompts.")

# Ensure the number of loaded images matches the number of prompts
assert len(images) == len(prompts), "Mismatch between number of images and prompts"

# Prepare prompts using LLaVA processor
processed_prompts = [llava_processor.apply_chat_template(prompt, add_generation_prompt=True) for prompt in prompts]

print("Prompts are processed and ready for generation.")


Number of queries in the query_generative.json:  1004
Selected queries:  [{'id': 865, 'image': 'AMBER_865.jpg', 'query': 'Describe this image.'}, {'id': 395, 'image': 'AMBER_395.jpg', 'query': 'Describe this image.'}, {'id': 777, 'image': 'AMBER_777.jpg', 'query': 'Describe this image.'}, {'id': 912, 'image': 'AMBER_912.jpg', 'query': 'Describe this image.'}, {'id': 431, 'image': 'AMBER_431.jpg', 'query': 'Describe this image.'}, {'id': 42, 'image': 'AMBER_42.jpg', 'query': 'Describe this image.'}, {'id': 266, 'image': 'AMBER_266.jpg', 'query': 'Describe this image.'}, {'id': 989, 'image': 'AMBER_989.jpg', 'query': 'Describe this image.'}, {'id': 524, 'image': 'AMBER_524.jpg', 'query': 'Describe this image.'}, {'id': 498, 'image': 'AMBER_498.jpg', 'query': 'Describe this image.'}]
Loaded 10 images and 10 prompts.
Prompts are processed and ready for generation.


In [15]:
import re
llava_model = model
# Prepare inputs for generation using LLaVA processor
inputs = llava_processor(
    images=images,
    text=processed_prompts,
    padding=True,
    return_tensors="pt"
).to("cuda:0", torch.float16)  # Send inputs to GPU0

# Generate descriptions with the LLaVA model
with torch.no_grad():
    generated_ids = llava_model.generate(**inputs, max_new_tokens=50)

# Decode the generated descriptions
generated_descriptions = llava_processor.batch_decode(generated_ids, skip_special_tokens=True)

# Store the results (image and generated description pairs)
generated_results = []

# Regular expression to remove templates like "[INST] ... [/INST]" and the content between them
template_pattern = re.compile(r"\[INST\].*?\[/INST\]", re.DOTALL)

for query, description in zip(selected_queries, generated_descriptions):
    # Remove the prompt templates from the generated output
    cleaned_description = re.sub(template_pattern, "", description).strip()

    # Store only the cleaned response
    generated_results.append({
        "id": query["id"],
        "image": query["image"],
        "original_prompt": query["query"],
        "generated_description": cleaned_description
    })

# Print generated descriptions for verification
for result in generated_results:
    print(f"Image ID: {result['id']}, Description: {result['generated_description']}")

# Save the generated results into a JSON file for further processing
output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/generated_results.json"
with open(output_path, "w") as f:
    json.dump(generated_results, f, indent=4)

print(f"Generated descriptions saved to {output_path}")


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Image ID: 865, Description: The image shows a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has
Image ID: 395, Description: In the tranquil expanse of a verdant field, a group of six sheep, their coats a mix of white and brown, are scattered across the landscape. The field, a vibrant green, is dotted with patches of dirt
Image ID: 777, Description: In the image, a black cat with a red collar is the main subject. The cat is standing on a gray concrete surface, its body facing away from the camera, and its head turned to the left. The cat's tail is raised,
Image ID: 912, Description: The image shows a tree with green leaves and a blue sky in the background. A boat is visible on the water, and the tree appears to be growing in the foreground. The image is a photograph with a focus on the tree and the boat
Image ID: 431, Description: In th

In [16]:
from transformers import CLIPTextModel, CLIPTokenizer
import torch.nn.functional as F

# Prepare inputs for the CLIP model
clip_text_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
clip_text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda:1")

hallucination_results = []

for result in generated_results:
    # Load the image
    image_path = os.path.join(dataset_dir, result["image"])
    image = Image.open(image_path)

    # Process the image for CLIP model
    processed_image = clip_processor(images=image, return_tensors="pt").to("cuda:1")

    # Process the generated description
    text_inputs = clip_text_tokenizer(result["generated_description"], return_tensors="pt", padding=True).to("cuda:1")

    # Get the CLIP embeddings
    with torch.no_grad():
        image_features = clip_model.get_image_features(**processed_image)
        text_features = clip_text_model(**text_inputs).pooler_output

    # Normalize the features
    image_features = F.normalize(image_features, p=2, dim=-1)
    text_features = F.normalize(text_features, p=2, dim=-1)

    # Calculate similarity score
    similarity_score = torch.matmul(image_features, text_features.T).item()

    # Determine if there is hallucination
    threshold = 0.25  # This threshold can be adjusted based on your requirements
    hallucinated = similarity_score*100 < threshold

    # Store the results
    hallucination_results.append({
        "id": result["id"],
        "image": result["image"],
        "original_prompt": result["original_prompt"],
        "generated_description": result["generated_description"],
        "similarity_score": similarity_score,
        "hallucinated": hallucinated
    })

    # Print results for verification
    print(f"Image ID: {result['id']}, Similarity Score: {similarity_score}, Hallucinated: {hallucinated}")

# Save the hallucination check results to a JSON file
hallucination_output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json"
with open(hallucination_output_path, "w") as f:
    json.dump(hallucination_results, f, indent=4)

print(f"Hallucination results saved to {hallucination_output_path}")


Image ID: 865, Similarity Score: 0.010856084525585175, Hallucinated: False
Image ID: 395, Similarity Score: 0.04918362945318222, Hallucinated: False
Image ID: 777, Similarity Score: -0.07679982483386993, Hallucinated: True
Image ID: 912, Similarity Score: -0.0837477371096611, Hallucinated: True
Image ID: 431, Similarity Score: -0.016975849866867065, Hallucinated: True
Image ID: 42, Similarity Score: 0.061216846108436584, Hallucinated: False
Image ID: 266, Similarity Score: -0.017193753272294998, Hallucinated: True
Image ID: 989, Similarity Score: -0.054267171770334244, Hallucinated: True
Image ID: 524, Similarity Score: 0.02029609866440296, Hallucinated: False
Image ID: 498, Similarity Score: 0.07267353683710098, Hallucinated: False
Hallucination results saved to /home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json


In [ ]:
import re

# Set the maximum number of regeneration attempts
max_rounds = 3

# Choose the refinement strategy manually: "strategy_1" or "strategy_2"
refinement_strategy = "strategy_2"  # Change this to "strategy_1" for the alternative strategy

final_results = []

# Iterate over the hallucination results and regenerate if hallucinated
for result in hallucination_results:
    current_image_path = os.path.join(dataset_dir, result["image"])
    image = Image.open(current_image_path)
    original_prompt = result["original_prompt"]  # Original prompt to be appended
    current_prompt = result["generated_description"]  # This is the initial generated description
    similarity_scores = [result["similarity_score"]]
    descriptions = [current_prompt]
    hallucination_flags = [result["hallucinated"]]
    rounds_info = []

    hallucinated = result["hallucinated"]

    # Loop for regeneration up to max_rounds if hallucinated
    for round_num in range(max_rounds):
        # Define the refined prompt based on the selected strategy
        if refinement_strategy == "strategy_1":
            # Strategy 1: Prefix warning about object, attribute, and relationship hallucinations
            refined_prompt = (
                f"Please be careful of object, attribute, and relationship hallucinations. Now, "
                f"{original_prompt}"
            )
        elif refinement_strategy == "strategy_2":
            # Strategy 2: Feedback on the last response and prefix warning
            refined_prompt = (
                f"There are hallucinations in the last response: '{current_prompt}'. "
                f"Please be careful of object, attribute, and relationship hallucinations in your response. Now, "
                f"{original_prompt}"

            )
        elif refinement_strategy == "strategy_3":
            # Strategy 3: Prefix warning about object, attribute, and relationship hallucinations, with feedback on the generation process
            refined_prompt = (
                f"Please be careful of object, attribute, and relationship hallucinations. Now, "
                f"{original_prompt}"
            )
        else:
            raise ValueError("Invalid refinement strategy. Please choose 'strategy_1' or 'strategy_2'.")

        # Append the original prompt content to the refined prompt
        # refined_prompt += original_prompt
        
        # Construct prompt data for LLaVA model
        prompt_data = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": refined_prompt},
                ],
            },
        ]

        # Apply the chat template to create the input that will be sent to the model
        processed_prompt = llava_processor.apply_chat_template(prompt_data, add_generation_prompt=True)
        print(f"Refined prompt for round {round_num + 1}: {processed_prompt}")
        if refinement_strategy == "strategy_3":
            processed_prompt +=  f"The last response was: '{current_prompt}' and it is inaccurate. The new response is: "


        # Prepare inputs for LLaVA generation
        inputs = llava_processor(
            images=image,
            text=processed_prompt,
            padding=True,
            return_tensors="pt"
        ).to("cuda:0", torch.float16)

        # Generate new description
        with torch.no_grad():
            generated_ids = llava_model.generate(**inputs, max_new_tokens=500, temperature=1.5, top_k=50 ,top_p=0.95)
        regenerated_description = llava_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        print(f"Round {round_num + 1}: Image ID: {result['id']}, Description: {regenerated_description}")

        # Remove prompt template content from the regenerated description
        # Remove prompt template content from the regenerated description
        if "[/INST]" in regenerated_description:
            cleaned_description = re.sub(template_pattern, "", regenerated_description).strip()
        else:
            cleaned_description = regenerated_description.strip()
        if refinement_strategy == "strategy_3":
            cleaned_description = cleaned_description.split("The new response is: ")[1]
        current_prompt = cleaned_description
        # Store the regenerated description and the current round information
        descriptions.append(cleaned_description)
        hallucination_flags.append(hallucinated)

        # Pass the image and regenerated description to CLIP for verification
        text_inputs = clip_text_tokenizer(
            current_prompt,
            return_tensors="pt",
            padding="max_length",
            max_length=77,  # Limit to the maximum length supported by CLIP
            truncation=True
        ).to("cuda:1")
        
        processed_image = clip_processor(images=image, return_tensors="pt").to("cuda:1")

        # Get similarity score from CLIP model
        with torch.no_grad():
            image_features = clip_model.get_image_features(**processed_image)
            text_features = clip_text_model(**text_inputs).pooler_output

        # Normalize the features
        image_features = F.normalize(image_features, p=2, dim=-1)
        text_features = F.normalize(text_features, p=2, dim=-1)

        # Calculate similarity score
        similarity_score = torch.matmul(image_features, text_features.T).item()
        similarity_scores.append(similarity_score)

        # Determine if hallucinated
        hallucinated = similarity_score < 0.25

        # Store current round info, including the input prompt and unprocessed output
        rounds_info.append({
            "round": round_num + 1,
            "input_prompt": processed_prompt,  # Store the final version of the input prompt after applying the template
            "original_output": regenerated_description,  # Store the original generated description without cleaning
            "description": cleaned_description,  # Store the cleaned description
            "similarity_score": similarity_score,
            "hallucinated": hallucinated
        })

        # Log the current round's result
        print(f"Round {round_num + 1}: Image ID: {result['id']}, Similarity Score: {similarity_score}, Hallucinated: {hallucinated}")

        # If no hallucination, stop further regeneration
        if not hallucinated:
            break

    # After max_rounds or successful verification, determine the final response
    if hallucinated:
        # Select the description with the highest similarity score if still hallucinated
        best_index = similarity_scores.index(max(similarity_scores))
        selection_reason = "highest_similarity_score"
    else:
        # If verified successfully, use the latest description
        best_index = len(descriptions) - 1
        selection_reason = "hallucination_detection_false"

    # Store the final result along with all the intermediate rounds information
    final_results.append({
        "id": result["id"],
        "image": result["image"],
        "initial_generated_prompt": result["generated_description"],
        "final_description": descriptions[best_index],
        "similarity_scores": similarity_scores,
        "hallucination_flags": hallucination_flags,
        "rounds_info": rounds_info,
        "selection_reason": selection_reason,
        "final_round_index": best_index  # Store the index of the round chosen as the final result
    })

# Save the final results into a JSON file
final_output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/final_results_detailed.json"
with open(final_output_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"Final results saved to {final_output_path}")


Refined prompt for round 1: [INST] <image>
There are hallucinations in the last response: 'The image shows a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST]


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Round 1: Image ID: 865, Description: [INST]  
There are hallucinations in the last response: 'The image shows a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST] The image depicts a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has a cylindrical shape with multiple levels, and it is topped with a dome-like structure. The sky is devoid of any other clouds or objects, and the overall scene is serene and clear. 
Round 1: Image ID: 865, Similarity Score: 0.027551598846912384, Hallucinated: True
Refined prompt for round 2: [INST] <image>
There are hallucinations in t

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Round 2: Image ID: 865, Description: [INST]  
There are hallucinations in the last response: 'The image depicts a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has a cylindrical shape with multiple levels, and it is topped with a dome-like structure. The sky is devoid of any other clouds or objects, and the overall scene is serene and clear.'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST] The image shows a clear blue sky with a large, fluffy white cloud in the center. Below the cloud, there is a structure that appears to be a tall building with a reflective surface, possibly a skyscraper. The building has a cylindrical shape with multiple levels, and it is topped with a dome-like structure. The sky is devoid of any other clouds or objects, and the overall sce

...

Round 3: Image ID: 989, Description: [INST]  
There are hallucinations in the last response: 'The image shows a round, white cake with a single, lit candle in the center. The cake is placed on a white plate, and the background is blurred but appears to be a wooden surface. On top of the cake, there are strawberries, which are red in color. The strawberries are arranged in a circular pattern around the cake, and they are whole, with their green leaves still attached. The candle is a traditional birthday candle, with a wick at the top and a flame that is currently lit. The overall scene suggests a celebration, possibly a birthday, with the cake and candle as the centerpiece.'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST] The image depicts a round, white cake with a single, lit candle in the center. The cake is placed on a white plate, and the background is blurred but appears to be a wooden surface. On top of 

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Round 3: Image ID: 989, Similarity Score: -0.06056401506066322, Hallucinated: True
Refined prompt for round 1: [INST] <image>
There are hallucinations in the last response: 'In the heart of a serene park, a young man is immersed in a game of tennis. He's dressed in a blue shirt and plaid pants, holding a tennis racket in his right hand. His gaze is directed up'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST]


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Round 1: Image ID: 524, Description: [INST]  
There are hallucinations in the last response: 'In the heart of a serene park, a young man is immersed in a game of tennis. He's dressed in a blue shirt and plaid pants, holding a tennis racket in his right hand. His gaze is directed up'. Please be careful of object, attribute, and relationship hallucinations in your response. Now, Describe this image. [/INST] In the image, there is a young man standing in a grassy area of a park. He is dressed in a blue shirt and plaid pants, and he is holding a tennis racket in his right hand. His gaze is directed upwards, possibly towards the sky or a ball. The park is lush and green, with trees and grass visible in the background. There is a bench to the left of the man, and a car is parked on the right side of the image. The man appears to be alone in the park, and there are no other people or objects visible in the image. The overall atmosphere of the image is peaceful and serene. 
Round 1: Image ID: 

In [32]:
import json

# Load the hallucination results to inspect keys
hallucination_results_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json"
with open(hallucination_results_path, "r") as f:
    hallucination_results = json.load(f)

# Print the first result to inspect the keys
print(hallucination_results[0].keys())

dict_keys(['id', 'image', 'original_prompt', 'generated_description', 'similarity_score', 'hallucinated'])
